# Iterative Burst Detector Synthetic Validation

This notebook generates synthetic spike-train cultures with known ground-truth burst and silence intervals, runs the iterative burst detector, and visualizes whether detected events match the simulated behavior.

Success criteria:
- Independent Poisson cultures should produce few or no network bursts.
- Cascade-propagation cultures should produce network bursts near injected burst epochs.
- Unit count, burst length, burst timing, recruitment strength, and burst morphology should vary across simulated cultures.
- Composite cultures with multiple burst types should expose detector behavior when one well contains heterogeneous network events.
- Quantification tables should report detection precision/recall/F1, temporal overlap, timing error, and detector quality columns such as LLR, composite score, and FF.


In [ ]:
from __future__ import annotations
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from dataclasses import dataclass
from pathlib import Path
import sys
import os
import tempfile


import numpy as np
import pandas as pd

# Make imports work whether the notebook is launched from the repo root or notebooks/.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "pipeline_tasks").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError("Could not locate repository root containing pipeline_tasks/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline_tasks.analysis import (
    BurstDetectorConfig,
    IterativeBurstConfig,
    compute_iterative_bursts,
    compute_network_bursts,
)
from pipeline_tasks.analysis.burst_detector import BurstDetectorError
from pipeline_tasks.analysis.iterative_burst_detector import IterativeBurstError





## Configuration

Edit the values in the cell below before running. All other cells read from these variables.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                     CONFIGURATION                           ║
# ║  Edit these values before running the notebook.             ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Internal constants (advanced) ────────────────────────────────────────────
BASE_SEED            = 20260510  # RNG seed for all synthetic datasets
DEFAULT_DURATION_S   = 150.0    # duration of each synthetic recording (s)
MIN_SYNTHETIC_UNITS  = 20        # min units in a synthetic network
MAX_SYNTHETIC_UNITS  = 150       # max units in a synthetic network

# ── Data sources ─────────────────────────────────────────────────────────────
USE_SIMULATED_DATA   = False   # True → include synthetic benchmarks
REAL_DATA_ROOT       = repo_root / "data" / "real_sorted_data"
REAL_KILOSORT_LABELS = {"good"}   # set None to include every Kilosort cluster
REAL_DATASET_DESCRIPTION = "Real spikesorted spike times; no ground-truth burst labels."

# All generated figures are saved here as PNG files.
FIGURE_OUTPUT_DIR = repo_root / "output" / "nb06_validation_figures"
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {FIGURE_OUTPUT_DIR}")

PLOTLY_OFFLINE = False  # True → embed Plotly JS (~3 MB); False → load from CDN

# ── Detector configurations ───────────────────────────────────────────────────
iterative_config = IterativeBurstConfig(
    min_burst_modulation=0.1,
    cluster_events=True,
    cluster_initial_components=6,
)  # burstlet gate + GMM enabled
iterative_config_old = IterativeBurstConfig(
    min_burst_modulation=0.0,   # disable burst-modulation gate
    cluster_events=False,        # disable GMM event clustering
)  # reproduces pre-fix behaviour for before/after comparison
traditional_config = BurstDetectorConfig()
MIN_OVERLAP_S = 0.03  # minimum overlap (s) to count a detected event as a match

# ── Visualisation selectors ───────────────────────────────────────────────────
METHOD_FOR_RASTER = "iterative"   # "iterative" | "traditional"
METHOD_FOR_TABLE  = "iterative"
DATASET_FOR_TABLE = "network-medium-post-burst-silence"


## Synthetic And Optional Real Data Model

Independent datasets use per-unit Poisson spike trains only. Connected datasets include cascade, synchronous, long-wave, partial-recruitment, doublet, clustered, irregular, and post-burst-silence morphologies. Several datasets randomize unit count and burst parameters with fixed seeds for reproducibility. The composite culture intentionally combines many burst types in one recording so the detector must handle heterogeneous event shapes within the same well.

To add real spikesorted data, set `REAL_DATA_ROOT` in the setup cell to a folder containing Kilosort output directories or `curated_spike_times.npy` files. Set `USE_SIMULATED_DATA = False` to run only real data. Real data is appended as unlabeled datasets, so detector calls are shown and summarized but are not scored as false positives against ground truth.


In [ ]:

@dataclass
class SyntheticDataset:
    name: str
    culture_type: str
    description: str
    spike_times: dict[str, np.ndarray]
    duration_s: float
    burst_intervals: list[tuple[float, float]]
    silence_intervals: list[tuple[float, float]]
    params: dict


def merge_intervals(intervals: list[tuple[float, float]], duration_s: float) -> list[tuple[float, float]]:
    clipped = sorted((max(0.0, s), min(duration_s, e)) for s, e in intervals if e > 0 and s < duration_s)
    if not clipped:
        return []
    merged = [clipped[0]]
    for s, e in clipped[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def complement_intervals(duration_s: float, blocked: list[tuple[float, float]]) -> list[tuple[float, float]]:
    blocked = merge_intervals(blocked, duration_s)
    allowed = []
    cursor = 0.0
    for s, e in blocked:
        if s > cursor:
            allowed.append((cursor, s))
        cursor = max(cursor, e)
    if cursor < duration_s:
        allowed.append((cursor, duration_s))
    return allowed


def poisson_spikes_in_intervals(rate_hz: float, intervals: list[tuple[float, float]], rng: np.random.Generator) -> np.ndarray:
    if rate_hz <= 0 or not intervals:
        return np.array([], dtype=float)
    pieces = []
    for s, e in intervals:
        n = rng.poisson(rate_hz * max(0.0, e - s))
        if n:
            pieces.append(rng.uniform(s, e, n))
    if not pieces:
        return np.array([], dtype=float)
    return np.sort(np.concatenate(pieces).astype(float))


def make_unit_ids(n_units: int) -> list[str]:
    return [f"unit_{i:03d}" for i in range(n_units)]


def as_per_burst_values(value: float | list[float], n_bursts: int) -> list[float]:
    if isinstance(value, list):
        if len(value) != n_bursts:
            raise ValueError(f"Expected {n_bursts} per-burst values, got {len(value)}")
        return [float(v) for v in value]
    return [float(value)] * n_bursts


def heterogeneous_rates(
    rng: np.random.Generator,
    n_units: int,
    low_hz: float,
    high_hz: float,
    profile: str = "log_uniform",
) -> np.ndarray:
    if not (MIN_SYNTHETIC_UNITS <= n_units <= MAX_SYNTHETIC_UNITS):
        raise ValueError(f"n_units must be {MIN_SYNTHETIC_UNITS}-{MAX_SYNTHETIC_UNITS}, got {n_units}")
    if low_hz <= 0 or high_hz <= low_hz:
        raise ValueError("Expected 0 < low_hz < high_hz")
    if profile == "log_uniform":
        rates = np.exp(rng.uniform(np.log(low_hz), np.log(high_hz), n_units))
    elif profile == "bimodal":
        low_count = n_units // 2
        low_rates = np.exp(rng.uniform(np.log(low_hz), np.log(np.sqrt(low_hz * high_hz)), low_count))
        high_rates = np.exp(rng.uniform(np.log(np.sqrt(low_hz * high_hz)), np.log(high_hz), n_units - low_count))
        rates = np.concatenate([low_rates, high_rates])
        rng.shuffle(rates)
    else:
        raise ValueError(f"Unknown rate profile: {profile}")
    return rates.astype(float)


def rate_summary_params(rates_hz: np.ndarray, prefix: str = "baseline") -> dict:
    return {
        f"{prefix}_rates_hz": np.round(rates_hz, 3).tolist(),
        f"{prefix}_rate_mean_hz": float(np.mean(rates_hz)),
        f"{prefix}_rate_median_hz": float(np.median(rates_hz)),
        f"{prefix}_rate_min_hz": float(np.min(rates_hz)),
        f"{prefix}_rate_max_hz": float(np.max(rates_hz)),
    }


def unit_firing_rates(dataset: SyntheticDataset) -> dict[str, float]:
    return {unit: len(spikes) / dataset.duration_s for unit, spikes in dataset.spike_times.items()}


def units_sorted_by_firing_rate(dataset: SyntheticDataset, descending: bool = False) -> list[str]:
    rates = unit_firing_rates(dataset)
    return sorted(dataset.spike_times, key=lambda unit: (rates[unit], unit), reverse=descending)


def is_kilosort_dir(path: Path) -> bool:
    return path.is_dir() and all((path / filename).exists() for filename in ("spike_times.npy", "spike_clusters.npy", "params.py"))


def _resolve_real_spike_path(path: str | Path) -> Path:
    path = Path(path).expanduser()
    if path.is_file() or is_kilosort_dir(path):
        return path
    if not path.is_dir():
        raise FileNotFoundError(f"Real spike path does not exist: {path}")

    curated = sorted(path.rglob("curated_spike_times.npy"))
    kilosort_dirs = sorted({p.parent for p in path.rglob("spike_times.npy") if is_kilosort_dir(p.parent)})
    candidates = curated + kilosort_dirs
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(
            "Multiple real spike sources found. Use discover_real_spike_sources() or set REAL_DATA_ROOT to a root folder:\n"
            + "\n".join(str(p) for p in candidates[:20])
        )
    raise FileNotFoundError(
        f"Directory {path} does not contain curated_spike_times.npy or a Kilosort output "
        "with spike_times.npy, spike_clusters.npy, and params.py."
    )


def discover_real_spike_sources(root: str | Path | None) -> list[Path]:
    if root is None:
        return []
    root = Path(root).expanduser()
    if root.is_file() or is_kilosort_dir(root):
        return [root]
    if not root.is_dir():
        raise FileNotFoundError(f"REAL_DATA_ROOT does not exist: {root}")

    curated = sorted(root.rglob("curated_spike_times.npy"))
    kilosort_dirs = sorted({p.parent for p in root.rglob("spike_times.npy") if is_kilosort_dir(p.parent)})
    curated_parents = {p.parent for p in curated}
    sources = curated + [p for p in kilosort_dirs if p not in curated_parents]
    return sorted(set(sources), key=lambda p: str(p))


def real_dataset_name_from_path(source: Path, root: str | Path | None = REAL_DATA_ROOT) -> str:
    source = Path(source).expanduser()
    try:
        rel = source.relative_to(Path(root).expanduser()) if root is not None else source.name
    except ValueError:
        rel = source.name
    if isinstance(rel, Path):
        if source.name == "curated_spike_times.npy":
            rel = source.parent.relative_to(Path(root).expanduser()) if root is not None else source.parent.name
        parts = rel.parts
    else:
        parts = (str(rel),)
    safe = "-".join(parts).replace(" ", "_")
    safe = safe.replace("curated_spike_times.npy", "curated")
    return f"real-{safe}".strip("-")


def _read_kilosort_sample_rate(params_path: Path) -> float:
    values: dict[str, object] = {}
    exec(params_path.read_text(), {}, values)
    for key in ("sample_rate", "fs", "sampling_rate"):
        if key in values:
            return float(values[key])
    raise ValueError(f"Could not find sample_rate/fs/sampling_rate in {params_path}")


def _read_kilosort_keep_clusters(ks_dir: Path, labels: set[str] | None) -> set[int] | None:
    if labels is None:
        return None
    for filename in ("cluster_group.tsv", "cluster_KSLabel.tsv"):
        label_path = ks_dir / filename
        if not label_path.exists():
            continue
        table = pd.read_csv(label_path, sep="\t")
        if "cluster_id" not in table.columns:
            continue
        label_col = next((col for col in ("group", "KSLabel", "label") if col in table.columns), None)
        if label_col is None:
            continue
        keep = table[table[label_col].astype(str).str.lower().isin({label.lower() for label in labels})]
        return set(keep["cluster_id"].astype(int).tolist())
    return None


def load_kilosort_spike_times(ks_dir: str | Path, labels: set[str] | None = REAL_KILOSORT_LABELS) -> dict[str, np.ndarray]:
    ks_dir = Path(ks_dir).expanduser()
    sample_rate = _read_kilosort_sample_rate(ks_dir / "params.py")
    spike_samples = np.load(ks_dir / "spike_times.npy").reshape(-1).astype(float)
    spike_clusters = np.load(ks_dir / "spike_clusters.npy").reshape(-1).astype(int)
    if spike_samples.shape[0] != spike_clusters.shape[0]:
        raise ValueError("Kilosort spike_times.npy and spike_clusters.npy have different lengths")

    keep_clusters = _read_kilosort_keep_clusters(ks_dir, labels)
    spike_times: dict[str, np.ndarray] = {}
    for cluster_id in np.unique(spike_clusters):
        if keep_clusters is not None and int(cluster_id) not in keep_clusters:
            continue
        times = spike_samples[spike_clusters == cluster_id] / sample_rate
        times = times[np.isfinite(times)]
        times = times[times >= 0.0]
        if times.size:
            spike_times[f"cluster_{int(cluster_id):03d}"] = np.sort(times.astype(float))
    if not spike_times:
        label_msg = "all clusters" if labels is None else f"labels {sorted(labels)}"
        raise ValueError(f"No non-empty Kilosort clusters found in {ks_dir} for {label_msg}")
    return spike_times


def load_curated_spike_times(path: str | Path) -> dict[str, np.ndarray]:
    path = _resolve_real_spike_path(path)
    if path.is_dir():
        return load_kilosort_spike_times(path)

    loaded = np.load(path, allow_pickle=True)
    if isinstance(loaded, np.lib.npyio.NpzFile):
        raw = {key: loaded[key] for key in loaded.files}
    elif isinstance(loaded, np.ndarray) and loaded.shape == () and isinstance(loaded.item(), dict):
        raw = loaded.item()
    elif isinstance(loaded, np.ndarray) and loaded.dtype == object and loaded.size == 1 and isinstance(loaded.item(), dict):
        raw = loaded.item()
    else:
        raise ValueError(
            "Expected curated spike times as a dict-like .npy/.npz: {unit_id: spike_times_seconds}."
        )

    spike_times: dict[str, np.ndarray] = {}
    for unit, times in raw.items():
        arr = np.asarray(times, dtype=float).reshape(-1)
        arr = arr[np.isfinite(arr)]
        arr = arr[arr >= 0.0]
        spike_times[str(unit)] = np.sort(arr)
    spike_times = {unit: times for unit, times in spike_times.items() if times.size > 0}
    if not spike_times:
        raise ValueError(f"No non-empty units found in {path}")
    return spike_times


def generate_real_curated_dataset(
    path: str | Path,
    name: str | None = None,
    description: str = REAL_DATASET_DESCRIPTION,
) -> SyntheticDataset:
    resolved_path = _resolve_real_spike_path(path)
    spike_times = load_curated_spike_times(resolved_path)
    duration_s = max(float(times[-1]) for times in spike_times.values() if times.size > 0)
    if duration_s <= 0:
        raise ValueError(f"Could not infer a positive recording duration from {resolved_path}")
    rates_hz = np.asarray([len(times) / duration_s for times in spike_times.values()], dtype=float)
    return SyntheticDataset(
        name=name or real_dataset_name_from_path(resolved_path),
        culture_type="real_spikesorted",
        description=description,
        spike_times=spike_times,
        duration_s=duration_s,
        burst_intervals=[],
        silence_intervals=[],
        params={
            "is_real_data": True,
            "source_path": str(resolved_path),
            "n_units": len(spike_times),
            **rate_summary_params(rates_hz),
        },
    )


def build_real_datasets(root: str | Path | None = REAL_DATA_ROOT) -> list[SyntheticDataset]:
    datasets = []
    seen_names: dict[str, int] = {}
    for source in discover_real_spike_sources(root):
        base_name = real_dataset_name_from_path(source, root)
        count = seen_names.get(base_name, 0)
        seen_names[base_name] = count + 1
        name = base_name if count == 0 else f"{base_name}-{count + 1}"
        datasets.append(generate_real_curated_dataset(source, name=name))
    return datasets


def generate_independent_poisson(
    name: str,
    description: str,
    rates_hz: np.ndarray,
    duration_s: float,
    seed: int,
) -> SyntheticDataset:
    rng = np.random.default_rng(seed)
    units = make_unit_ids(len(rates_hz))
    full_interval = [(0.0, duration_s)]
    spike_times = {
        unit: poisson_spikes_in_intervals(float(rate), full_interval, rng)
        for unit, rate in zip(units, rates_hz)
    }
    return SyntheticDataset(
        name=name,
        culture_type="independent_poisson",
        description=description,
        spike_times=spike_times,
        duration_s=duration_s,
        burst_intervals=[],
        silence_intervals=[],
        params={"n_units": len(rates_hz), **rate_summary_params(rates_hz)},
    )


def generate_cascade_culture(
    name: str,
    description: str,
    baseline_rates_hz: np.ndarray,
    burst_starts_s: list[float],
    burst_rate_hz: float,
    recruitment_probability: float,
    duration_s: float,
    seed: int,
    group_count: int = 4,
    group_delay_s: float = 0.08,
    group_burst_duration_s: float | list[float] = 0.50,
    burst_jitter_s: float | None = None,
    silence_after_s: float = 0.0,
    silence_rate_scale: float = 0.05,
) -> SyntheticDataset:
    rng = np.random.default_rng(seed)
    units = make_unit_ids(len(baseline_rates_hz))
    unit_groups = np.array_split(np.arange(len(units)), group_count)
    burst_durations = as_per_burst_values(group_burst_duration_s, len(burst_starts_s))

    cascade_spans = [(group_count - 1) * group_delay_s + dur for dur in burst_durations]
    burst_intervals = [(start, start + span) for start, span in zip(burst_starts_s, cascade_spans)]
    silence_intervals = [
        (start + span, start + span + silence_after_s)
        for start, span in zip(burst_starts_s, cascade_spans)
        if silence_after_s > 0
    ]
    blocked = merge_intervals(burst_intervals + silence_intervals, duration_s)
    background_intervals = complement_intervals(duration_s, blocked)

    spike_times: dict[str, list[float]] = {unit: [] for unit in units}

    for unit, rate in zip(units, baseline_rates_hz):
        spike_times[unit].extend(poisson_spikes_in_intervals(float(rate), background_intervals, rng).tolist())

    if silence_intervals:
        for unit, rate in zip(units, baseline_rates_hz):
            spike_times[unit].extend(
                poisson_spikes_in_intervals(float(rate) * silence_rate_scale, silence_intervals, rng).tolist()
            )

    for burst_start, burst_duration in zip(burst_starts_s, burst_durations):
        jitter = float(burst_jitter_s) if burst_jitter_s is not None else max(0.012, burst_duration / 7)
        for group_idx, group in enumerate(unit_groups):
            group_start = burst_start + group_idx * group_delay_s
            group_end = group_start + burst_duration
            for unit_idx in group:
                if rng.random() > recruitment_probability:
                    continue
                unit = units[int(unit_idx)]
                n_spikes = rng.poisson(burst_rate_hz * burst_duration)
                n_spikes = max(1, int(n_spikes))
                burst_spikes = rng.normal(
                    loc=(group_start + group_end) / 2,
                    scale=jitter,
                    size=n_spikes,
                )
                burst_spikes = burst_spikes[(burst_spikes >= group_start) & (burst_spikes < group_end)]
                if burst_spikes.size == 0:
                    burst_spikes = np.array([rng.uniform(group_start, group_end)])
                spike_times[unit].extend(burst_spikes.tolist())

    sorted_spikes = {
        unit: np.sort(np.asarray([s for s in spikes if 0.0 <= s < duration_s], dtype=float))
        for unit, spikes in spike_times.items()
    }
    return SyntheticDataset(
        name=name,
        culture_type="cascade_connected",
        description=description,
        spike_times=sorted_spikes,
        duration_s=duration_s,
        burst_intervals=merge_intervals(burst_intervals, duration_s),
        silence_intervals=merge_intervals(silence_intervals, duration_s),
        params={
            "n_units": len(baseline_rates_hz),
            **rate_summary_params(baseline_rates_hz),
            "burst_rate_hz": burst_rate_hz,
            "recruitment_probability": recruitment_probability,
            "n_bursts": len(burst_starts_s),
            "group_count": group_count,
            "group_delay_s": group_delay_s,
            "group_burst_duration_s": burst_durations,
            "truth_duration_s": np.round(cascade_spans, 3).tolist(),
            "silence_after_s": silence_after_s,
            "silence_rate_scale": silence_rate_scale,
        },
    )


def generate_composite_burst_culture(
    name: str,
    description: str,
    n_units: int,
    baseline_rates_hz: np.ndarray,
    burst_specs: list[dict],
    duration_s: float,
    seed: int,
) -> SyntheticDataset:
    rng = np.random.default_rng(seed)
    units = make_unit_ids(n_units)
    unit_indices = np.arange(n_units)
    spike_times: dict[str, list[float]] = {unit: [] for unit in units}

    truth_intervals: list[tuple[float, float]] = []
    silence_intervals: list[tuple[float, float]] = []
    for spec in burst_specs:
        start = float(spec["start_s"])
        burst_type = spec["type"]
        if burst_type == "doublet":
            duration = float(spec.get("duration_s", 0.50))
            separation = float(spec.get("separation_s", 0.80))
            end = start + separation + duration
        elif burst_type in {"cascade", "long_wave"}:
            group_count = int(spec.get("group_count", 4))
            group_delay = float(spec.get("group_delay_s", 0.08))
            duration = float(spec.get("duration_s", 0.70))
            end = start + (group_count - 1) * group_delay + duration
        else:
            duration = float(spec.get("duration_s", 0.70))
            end = start + duration
        truth_intervals.append((start, end))
        silence_after = float(spec.get("silence_after_s", 0.0))
        if silence_after > 0:
            silence_intervals.append((end, end + silence_after))

    blocked = merge_intervals(truth_intervals + silence_intervals, duration_s)
    background_intervals = complement_intervals(duration_s, blocked)

    for unit, rate in zip(units, baseline_rates_hz):
        spike_times[unit].extend(poisson_spikes_in_intervals(float(rate), background_intervals, rng).tolist())

    if silence_intervals:
        silence_scale = 0.03
        for unit, rate in zip(units, baseline_rates_hz):
            spike_times[unit].extend(
                poisson_spikes_in_intervals(float(rate) * silence_scale, silence_intervals, rng).tolist()
            )

    def add_window_spikes(indices: np.ndarray, start: float, end: float, rate_hz: float, recruitment: float) -> None:
        duration = max(0.0, end - start)
        jitter = max(0.012, duration / 7)
        for idx in indices:
            if rng.random() > recruitment:
                continue
            n_spikes = max(1, int(rng.poisson(rate_hz * duration)))
            spikes = rng.normal(loc=(start + end) / 2, scale=jitter, size=n_spikes)
            spikes = spikes[(spikes >= start) & (spikes < end)]
            if spikes.size == 0:
                spikes = np.array([rng.uniform(start, end)])
            spike_times[units[int(idx)]].extend(spikes.tolist())

    burst_types_used = []
    recruitments = []
    for spec in burst_specs:
        burst_type = spec["type"]
        burst_types_used.append(burst_type)
        start = float(spec["start_s"])
        rate = float(spec.get("burst_rate_hz", 42.0))
        recruitment = float(spec.get("recruitment_probability", 0.8))
        recruitments.append(recruitment)
        duration = float(spec.get("duration_s", 0.70))

        if burst_type == "synchronous":
            add_window_spikes(unit_indices, start, start + duration, rate, recruitment)
        elif burst_type == "partial":
            subnetwork_frac = float(spec.get("subnetwork_frac", 0.45))
            n_sub = max(1, int(round(n_units * subnetwork_frac)))
            subnetwork = rng.choice(unit_indices, size=n_sub, replace=False)
            add_window_spikes(subnetwork, start, start + duration, rate, recruitment)
        elif burst_type == "doublet":
            separation = float(spec.get("separation_s", 0.80))
            add_window_spikes(unit_indices, start, start + duration, rate, recruitment)
            add_window_spikes(unit_indices, start + separation, start + separation + duration, rate, recruitment)
        elif burst_type in {"cascade", "long_wave"}:
            group_count = int(spec.get("group_count", 4))
            group_delay = float(spec.get("group_delay_s", 0.08 if burst_type == "cascade" else 0.18))
            groups = np.array_split(unit_indices, group_count)
            for group_idx, group in enumerate(groups):
                group_start = start + group_idx * group_delay
                add_window_spikes(group, group_start, group_start + duration, rate, recruitment)
        else:
            raise ValueError(f"Unknown burst type: {burst_type}")

    sorted_spikes = {
        unit: np.sort(np.asarray([s for s in spikes if 0.0 <= s < duration_s], dtype=float))
        for unit, spikes in spike_times.items()
    }
    truth_durations = [e - s for s, e in merge_intervals(truth_intervals, duration_s)]
    return SyntheticDataset(
        name=name,
        culture_type="composite_connected",
        description=description,
        spike_times=sorted_spikes,
        duration_s=duration_s,
        burst_intervals=merge_intervals(truth_intervals, duration_s),
        silence_intervals=merge_intervals(silence_intervals, duration_s),
        params={
            "n_units": n_units,
            **rate_summary_params(baseline_rates_hz),
            "recruitment_probability": float(np.mean(recruitments)) if recruitments else np.nan,
            "truth_duration_s": np.round(truth_durations, 3).tolist(),
            "burst_types": burst_types_used,
            "burst_specs": burst_specs,
        },
    )


def unit_count_tier(n_units: int) -> str:
    if n_units <= 40:
        return "small (<=40)"
    if n_units <= 100:
        return "medium (41-100)"
    return "large (101-150)"


def baseline_rate_tier(dataset: SyntheticDataset) -> str:
    rates = np.asarray(dataset.params.get("baseline_rates_hz", []), dtype=float)
    if rates.size == 0:
        return "unknown"
    spread = float(np.max(rates) / max(np.min(rates), 1e-12))
    median = float(np.median(rates))
    if spread >= 20:
        return "mixed"
    if median < 0.35:
        return "low"
    if median < 1.2:
        return "medium"
    return "high"


def burst_duration_tier(dataset: SyntheticDataset) -> str:
    if not dataset.burst_intervals:
        return "none"
    durations = np.asarray([e - s for s, e in dataset.burst_intervals], dtype=float)
    if np.ptp(durations) >= 0.75:
        return "variable"
    median = float(np.median(durations))
    if median < 0.75:
        return "medium-short (0.5-0.75s)"
    if median < 1.25:
        return "medium (0.75-1.25s)"
    return "long (1.25-2.0s)"


def morphology_label(dataset: SyntheticDataset) -> str:
    if dataset.params.get("is_real_data", False):
        return "real spikesorted"
    name = dataset.name
    if dataset.culture_type == "independent_poisson":
        return "independent"
    if dataset.culture_type == "composite_connected":
        return "composite"
    for token, label in [
        ("clustered", "clustered"),
        ("irregular", "irregular"),
        ("weak-recruitment", "weak recruitment"),
        ("silence", "post-burst silence"),
        ("variable-duration", "variable duration"),
        ("long-duration", "long duration"),
        ("short-duration", "short duration"),
    ]:
        if token in name:
            return label
    return "cascade"


def recruitment_tier(dataset: SyntheticDataset) -> str:
    value = dataset.params.get("recruitment_probability")
    if value is None or pd.isna(value):
        return "none"
    value = float(value)
    if value < 0.55:
        return "weak"
    if value < 0.78:
        return "moderate"
    return "strong"


def dataset_category_row(dataset: SyntheticDataset) -> dict:
    if dataset.params.get("is_real_data", False):
        expected = "real data (unlabeled)"
    else:
        expected = "negative control" if not dataset.burst_intervals else "burst-positive"
    n_units = len(dataset.spike_times)
    morphology = morphology_label(dataset)
    duration_tier = burst_duration_tier(dataset)
    rate_tier = baseline_rate_tier(dataset)
    recruitment = recruitment_tier(dataset)
    return {
        "expected": expected,
        "unit_count_tier": unit_count_tier(n_units),
        "baseline_rate_tier": rate_tier,
        "burst_duration_tier": duration_tier,
        "morphology": morphology,
        "recruitment_tier": recruitment,
        "benchmark_group": f"{expected} | {unit_count_tier(n_units)} | {rate_tier} rate | {morphology}",
    }


In [ ]:

def build_synthetic_datasets() -> list[SyntheticDataset]:
    rng = np.random.default_rng(BASE_SEED)
    duration_s = DEFAULT_DURATION_S

    datasets = [
        generate_independent_poisson(
            "random-low-independent",
            "Independent low-rate Poisson units with heterogeneous baseline rates; expected to be mostly quiet with no network structure.",
            heterogeneous_rates(rng, 20, 0.04, 0.35),
            duration_s,
            seed=BASE_SEED + 1,
        ),
        generate_independent_poisson(
            "random-medium-independent",
            "Independent medium-rate Poisson units with heterogeneous baseline rates; random coincidences may occur but no shared drive exists.",
            heterogeneous_rates(rng, 45, 0.18, 1.6),
            duration_s,
            seed=BASE_SEED + 2,
        ),
        generate_independent_poisson(
            "random-high-independent",
            "Independent high-rate Poisson units with heterogeneous baseline rates; tests false positives under dense unstructured firing.",
            heterogeneous_rates(rng, 90, 0.75, 5.5),
            duration_s,
            seed=BASE_SEED + 3,
        ),
        generate_independent_poisson(
            "random-mixed-heterogeneous",
            "Independent heterogeneous Poisson units spanning very low to high individual rates.",
            heterogeneous_rates(rng, 150, 0.04, 6.0, profile="bimodal"),
            duration_s,
            seed=BASE_SEED + 4,
        ),
        generate_cascade_culture(
            "network-low-sparse-cascade",
            "Few weak cascade bursts on low heterogeneous background firing.",
            heterogeneous_rates(rng, 25, 0.05, 0.35),
            [30.0, 78.0, 126.0],
            burst_rate_hz=22.0,
            recruitment_probability=0.55,
            duration_s=duration_s,
            seed=BASE_SEED + 5,
            group_delay_s=0.10,
            group_burst_duration_s=0.50,
        ),
        generate_cascade_culture(
            "network-high-sparse-cascade",
            "Few cascade bursts on higher heterogeneous individual background rates.",
            heterogeneous_rates(rng, 50, 0.35, 2.4),
            [30.0, 78.0, 126.0],
            burst_rate_hz=32.0,
            recruitment_probability=0.65,
            duration_s=duration_s,
            seed=BASE_SEED + 6,
            group_delay_s=0.12,
            group_burst_duration_s=0.90,
        ),
        generate_cascade_culture(
            "network-medium-frequent-cascade",
            "Moderately frequent cascade bursts with heterogeneous medium individual firing rates.",
            heterogeneous_rates(rng, 75, 0.18, 1.4),
            [18.0, 42.0, 66.0, 90.0, 114.0, 138.0],
            burst_rate_hz=34.0,
            recruitment_probability=0.75,
            duration_s=duration_s,
            seed=BASE_SEED + 7,
            group_delay_s=0.08,
            group_burst_duration_s=0.65,
        ),
        generate_cascade_culture(
            "network-mixed-strong-cascade",
            "Strong frequent cascade bursts with very heterogeneous unit baselines.",
            heterogeneous_rates(rng, 120, 0.05, 3.5, profile="bimodal"),
            [16.0, 34.0, 52.0, 70.0, 88.0, 106.0, 124.0, 142.0],
            burst_rate_hz=42.0,
            recruitment_probability=0.90,
            duration_s=duration_s,
            seed=BASE_SEED + 8,
            group_delay_s=0.06,
            group_burst_duration_s=0.50,
        ),
        generate_cascade_culture(
            "network-medium-post-burst-silence",
            "Cascade bursts followed by a brief low-rate silence period across connected units.",
            heterogeneous_rates(rng, 80, 0.18, 1.5),
            [24.0, 58.0, 92.0, 126.0],
            burst_rate_hz=36.0,
            recruitment_probability=0.85,
            duration_s=duration_s,
            seed=BASE_SEED + 9,
            group_delay_s=0.08,
            group_burst_duration_s=0.75,
            silence_after_s=1.2,
            silence_rate_scale=0.03,
        ),
        generate_cascade_culture(
            "network-medium-short-duration",
            "Shortest allowed cascade bursts; tests temporal resolution and missed short events.",
            heterogeneous_rates(rng, 60, 0.16, 1.3),
            [20.0, 45.0, 70.0, 95.0, 120.0],
            burst_rate_hz=58.0,
            recruitment_probability=0.85,
            duration_s=duration_s,
            seed=BASE_SEED + 10,
            group_delay_s=0.05,
            group_burst_duration_s=0.35,
        ),
        generate_cascade_culture(
            "network-medium-long-duration",
            "Long cascade bursts near the two-second upper bound; tests duration recovery and over-trimming.",
            heterogeneous_rates(rng, 100, 0.16, 1.3),
            [20.0, 55.0, 90.0, 125.0],
            burst_rate_hz=20.0,
            recruitment_probability=0.90,
            duration_s=duration_s,
            seed=BASE_SEED + 11,
            group_delay_s=0.20,
            group_burst_duration_s=1.40,
        ),
        generate_cascade_culture(
            "network-medium-variable-duration",
            "Same culture with burst durations ranging from 0.5 to almost 2 seconds.",
            heterogeneous_rates(rng, 90, 0.16, 1.3),
            [18.0, 45.0, 72.0, 99.0, 126.0],
            burst_rate_hz=32.0,
            recruitment_probability=0.82,
            duration_s=duration_s,
            seed=BASE_SEED + 12,
            group_delay_s=0.075,
            group_burst_duration_s=[0.35, 0.60, 0.90, 1.20, 1.70],
        ),
        generate_cascade_culture(
            "network-low-weak-recruitment",
            "Low firing rate and weak recruitment; intentionally difficult positive control.",
            heterogeneous_rates(rng, 35, 0.05, 0.45),
            [24.0, 62.0, 100.0, 138.0],
            burst_rate_hz=28.0,
            recruitment_probability=0.38,
            duration_s=duration_s,
            seed=BASE_SEED + 13,
            group_delay_s=0.08,
            group_burst_duration_s=0.55,
        ),
        generate_cascade_culture(
            "network-medium-clustered-bursts",
            "Bursts arrive in close clusters; tests whether nearby bursts are split or merged.",
            heterogeneous_rates(rng, 70, 0.14, 1.4),
            [20.0, 20.9, 21.8, 78.0, 78.9, 79.8, 132.0, 132.9, 133.8],
            burst_rate_hz=34.0,
            recruitment_probability=0.78,
            duration_s=duration_s,
            seed=BASE_SEED + 14,
            group_delay_s=0.07,
            group_burst_duration_s=0.45,
        ),
        generate_cascade_culture(
            "network-high-irregular-intervals",
            "High-rate bursts at irregular intervals; tests adaptive background estimation.",
            heterogeneous_rates(rng, 110, 0.35, 2.6),
            [12.0, 27.5, 63.0, 74.0, 119.5, 146.0],
            burst_rate_hz=30.0,
            recruitment_probability=0.80,
            duration_s=duration_s,
            seed=BASE_SEED + 15,
            group_delay_s=0.12,
            group_burst_duration_s=0.85,
        ),
        generate_independent_poisson(
            "random-low-few-units",
            "Independent Poisson control with low but in-range unit count and heterogeneous firing.",
            heterogeneous_rates(rng, int(rng.integers(20, 31)), 0.08, 0.65),
            duration_s,
            seed=BASE_SEED + 16,
        ),
        generate_independent_poisson(
            "random-medium-many-units",
            "Independent Poisson control with high unit count and heterogeneous firing.",
            heterogeneous_rates(rng, int(rng.integers(115, 151)), 0.16, 1.7),
            duration_s,
            seed=BASE_SEED + 17,
        ),
        generate_cascade_culture(
            "network-medium-randomized-cascade-a",
            "Randomized unit count and burst parameters sampled from broad but plausible ranges.",
            heterogeneous_rates(rng, int(rng.integers(45, 91)), 0.12, 1.4),
            sorted(rng.uniform(14.0, 140.0, int(rng.integers(4, 8))).round(1).tolist()),
            burst_rate_hz=float(rng.uniform(28.0, 46.0)),
            recruitment_probability=float(rng.uniform(0.62, 0.90)),
            duration_s=duration_s,
            seed=BASE_SEED + 18,
            group_delay_s=float(rng.uniform(0.04, 0.14)),
            group_burst_duration_s=float(rng.uniform(0.45, 1.45)),
        ),
        generate_cascade_culture(
            "network-high-randomized-cascade-b",
            "Second randomized connected culture with different unit count and morphology.",
            heterogeneous_rates(rng, int(rng.integers(90, 151)), 0.25, 2.6),
            sorted(rng.uniform(12.0, 145.0, int(rng.integers(5, 9))).round(1).tolist()),
            burst_rate_hz=float(rng.uniform(24.0, 42.0)),
            recruitment_probability=float(rng.uniform(0.55, 0.86)),
            duration_s=duration_s,
            seed=BASE_SEED + 19,
            group_delay_s=float(rng.uniform(0.05, 0.18)),
            group_burst_duration_s=float(rng.uniform(0.55, 1.30)),
        ),
    ]

    composite_n_units = int(rng.integers(100, 151))
    datasets.append(
        generate_composite_burst_culture(
            "composite-mixed-many-burst-types",
            "One culture containing synchronous, cascade, long-wave, partial, doublet, and post-silence bursts.",
            n_units=composite_n_units,
            baseline_rates_hz=heterogeneous_rates(rng, composite_n_units, 0.08, 2.2, profile="bimodal"),
            burst_specs=[
                {"type": "synchronous", "start_s": 14.0, "duration_s": 0.50, "burst_rate_hz": 48.0, "recruitment_probability": 0.86},
                {"type": "cascade", "start_s": 34.0, "duration_s": 0.70, "group_delay_s": 0.08, "burst_rate_hz": 38.0, "recruitment_probability": 0.78},
                {"type": "long_wave", "start_s": 58.0, "duration_s": 1.28, "group_delay_s": 0.18, "burst_rate_hz": 22.0, "recruitment_probability": 0.86},
                {"type": "partial", "start_s": 86.0, "duration_s": 0.85, "burst_rate_hz": 42.0, "recruitment_probability": 0.82, "subnetwork_frac": 0.42},
                {"type": "doublet", "start_s": 111.0, "duration_s": 0.50, "separation_s": 0.80, "burst_rate_hz": 46.0, "recruitment_probability": 0.78},
                {"type": "cascade", "start_s": 136.0, "duration_s": 0.65, "group_delay_s": 0.07, "burst_rate_hz": 40.0, "recruitment_probability": 0.84, "silence_after_s": 1.0},
            ],
            duration_s=duration_s,
            seed=BASE_SEED + 20,
        )
    )

    return datasets


def build_benchmark_datasets() -> list[SyntheticDataset]:
    datasets = build_synthetic_datasets() if USE_SIMULATED_DATA else []
    real_sources = discover_real_spike_sources(REAL_DATA_ROOT)
    if REAL_DATA_ROOT is not None and not real_sources:
        print(f"Warning: no real data sources found under {REAL_DATA_ROOT}")
    elif real_sources:
        print(f"Discovered {len(real_sources)} real data source(s) under {REAL_DATA_ROOT}:")
        for s in real_sources:
            print(f"  {s}")
    datasets.extend(build_real_datasets(REAL_DATA_ROOT))
    n_synthetic = sum(1 for d in datasets if not d.params.get('is_real_data'))
    n_real = sum(1 for d in datasets if d.params.get('is_real_data'))
    print(f"\nTotal datasets: {len(datasets)}  ({n_synthetic} synthetic, {n_real} real)")
    if not datasets:
        raise ValueError("No datasets configured. Enable USE_SIMULATED_DATA or set REAL_DATA_ROOT to real data.")
    return datasets


datasets = build_benchmark_datasets()

pd.DataFrame([
    {
        "dataset": d.name,
        "type": d.culture_type,
        "units": len(d.spike_times),
        "duration_s": d.duration_s,
        "truth_bursts": len(d.burst_intervals),
        "truth_silences": len(d.silence_intervals),
        "mean_truth_duration_s": np.mean([e - s for s, e in d.burst_intervals]) if d.burst_intervals else 0.0,
        "min_truth_duration_s": np.min([e - s for s, e in d.burst_intervals]) if d.burst_intervals else 0.0,
        "max_truth_duration_s": np.max([e - s for s, e in d.burst_intervals]) if d.burst_intervals else 0.0,
        "baseline_rate_min_hz": d.params.get("baseline_rate_min_hz", np.nan),
        "baseline_rate_median_hz": d.params.get("baseline_rate_median_hz", np.nan),
        "baseline_rate_max_hz": d.params.get("baseline_rate_max_hz", np.nan),
        "mean_spikes_per_unit": np.mean([len(v) for v in d.spike_times.values()]),
        "burst_types": ", ".join(d.params.get("burst_types", [])),
        "source_path": d.params.get("source_path", ""),
        **dataset_category_row(d),
        "description": d.description,
    }
    for d in datasets
])


## Run Both Detectors and Quantify Results

This section runs the new iterative detector and the traditional participation-based detector on the same synthetic datasets. Edit `iterative_config` or `traditional_config` here when testing parameter sensitivity.

The Poisson-only datasets are negative controls: any network-burst calls there should be treated as false positives to investigate.

Quantification includes event-level matching against ground truth plus summaries of detector output columns. A predicted event is counted as matched when it overlaps an unmatched truth burst by at least `MIN_OVERLAP_S`.

In [ ]:
def interval_overlap(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))


def interval_union(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(a[1], b[1]) - min(a[0], b[0])


def match_events_to_truth(events: pd.DataFrame, truth: list[tuple[float, float]], min_overlap_s: float = MIN_OVERLAP_S) -> list[dict]:
    if events.empty or not truth:
        return []
    event_intervals = list(zip(events["start"].astype(float), events["end"].astype(float)))
    candidates = []
    for event_idx, event_interval in enumerate(event_intervals):
        for truth_idx, truth_interval in enumerate(truth):
            overlap = interval_overlap(event_interval, truth_interval)
            if overlap >= min_overlap_s:
                candidates.append((overlap, event_idx, truth_idx, event_interval, truth_interval))
    candidates.sort(reverse=True, key=lambda x: x[0])

    used_events = set()
    used_truth = set()
    matches = []
    for overlap, event_idx, truth_idx, event_interval, truth_interval in candidates:
        if event_idx in used_events or truth_idx in used_truth:
            continue
        used_events.add(event_idx)
        used_truth.add(truth_idx)
        union = interval_union(event_interval, truth_interval)
        matches.append({
            "event_idx": event_idx,
            "truth_idx": truth_idx,
            "overlap_s": overlap,
            "iou": overlap / union if union > 0 else 0.0,
            "onset_error_s": event_interval[0] - truth_interval[0],
            "offset_error_s": event_interval[1] - truth_interval[1],
            "duration_error_s": (event_interval[1] - event_interval[0]) - (truth_interval[1] - truth_interval[0]),
        })
    return matches


def summarize_event_quality(events: pd.DataFrame) -> dict:
    quality_cols = [
        "duration_s", "peak_synchrony", "participation", "total_spikes",
        "llr_aggregate", "composite_peak", "composite_mean", "ff_peak",
    ]
    if events.empty:
        return {f"{col}_median": 0.0 for col in quality_cols} | {f"{col}_mean": 0.0 for col in quality_cols}
    summary = {}
    for col in quality_cols:
        if col in events:
            summary[f"{col}_median"] = float(events[col].median())
            summary[f"{col}_mean"] = float(events[col].mean())
    return summary


def interval_total_duration(intervals: list[tuple[float, float]]) -> float:
    return float(sum(max(0.0, e - s) for s, e in intervals))


def score_events_against_truth(events: pd.DataFrame, truth: list[tuple[float, float]], duration_s: float) -> dict:
    event_intervals = [] if events.empty else list(zip(events["start"].astype(float), events["end"].astype(float)))
    matches = match_events_to_truth(events, truth)
    matched_event_indices = {m["event_idx"] for m in matches}
    matched_truth_indices = {m["truth_idx"] for m in matches}
    true_positive = len(matches)
    false_positive = len(event_intervals) - true_positive
    false_negative = len(truth) - true_positive
    precision = true_positive / len(event_intervals) if event_intervals else (1.0 if not truth else 0.0)
    recall = true_positive / len(truth) if truth else (1.0 if not event_intervals else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    truth_time_s = interval_total_duration(truth)
    predicted_time_s = interval_total_duration(event_intervals)
    matched_overlap_s = float(sum(m["overlap_s"] for m in matches))
    false_positive_time_s = sum(
        (e - s) for idx, (s, e) in enumerate(event_intervals)
        if idx not in matched_event_indices
    )
    missed_truth_time_s = sum(
        (e - s) for idx, (s, e) in enumerate(truth)
        if idx not in matched_truth_indices
    )
    coverage_recall = matched_overlap_s / truth_time_s if truth_time_s > 0 else (1.0 if predicted_time_s == 0 else 0.0)
    coverage_precision = matched_overlap_s / predicted_time_s if predicted_time_s > 0 else (1.0 if truth_time_s == 0 else 0.0)
    coverage_f1 = (
        2 * coverage_precision * coverage_recall / (coverage_precision + coverage_recall)
        if coverage_precision + coverage_recall > 0 else 0.0
    )
    count_error = len(event_intervals) - len(truth)
    count_abs_error = abs(count_error)
    count_ratio = len(event_intervals) / len(truth) if truth else (0.0 if not event_intervals else float("inf"))
    event_rate_per_min = len(event_intervals) / (duration_s / 60.0) if duration_s > 0 else 0.0
    truth_rate_per_min = len(truth) / (duration_s / 60.0) if duration_s > 0 else 0.0
    false_positive_rate_per_min = false_positive / (duration_s / 60.0) if duration_s > 0 else 0.0
    missed_rate = false_negative / len(truth) if truth else 0.0
    fragmentation_index = len(event_intervals) / true_positive if true_positive > 0 else 0.0
    merge_flag = any(m["duration_error_s"] > 0.5 for m in matches)
    split_or_miss_flag = false_negative > 0 or fragmentation_index > 1.5

    return {
        "truth_bursts": len(truth),
        "detected_network_bursts": len(event_intervals),
        "true_positive_events": true_positive,
        "false_positive_events": false_positive,
        "false_negative_events": false_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_iou": float(np.mean([m["iou"] for m in matches])) if matches else 0.0,
        "median_abs_onset_error_s": float(np.median(np.abs([m["onset_error_s"] for m in matches]))) if matches else 0.0,
        "median_abs_offset_error_s": float(np.median(np.abs([m["offset_error_s"] for m in matches]))) if matches else 0.0,
        "median_abs_duration_error_s": float(np.median(np.abs([m["duration_error_s"] for m in matches]))) if matches else 0.0,
        "truth_time_s": truth_time_s,
        "predicted_time_s": predicted_time_s,
        "matched_overlap_s": matched_overlap_s,
        "coverage_precision": coverage_precision,
        "coverage_recall": coverage_recall,
        "coverage_f1": coverage_f1,
        "false_positive_time_s": false_positive_time_s,
        "missed_truth_time_s": missed_truth_time_s,
        "count_error": count_error,
        "count_abs_error": count_abs_error,
        "count_ratio": count_ratio,
        "truth_rate_per_min": truth_rate_per_min,
        "event_rate_per_min": event_rate_per_min,
        "false_positive_rate_per_min": false_positive_rate_per_min,
        "missed_rate": missed_rate,
        "fragmentation_index": fragmentation_index,
        "merge_flag": merge_flag,
        "split_or_miss_flag": split_or_miss_flag,
    }


def build_event_quality_table(dataset: SyntheticDataset, result) -> pd.DataFrame:
    events = result.network_bursts.copy()
    if events.empty:
        return pd.DataFrame()
    matches = match_events_to_truth(events, dataset.burst_intervals)
    matched_event_indices = {m["event_idx"] for m in matches}
    match_by_event = {m["event_idx"]: m for m in matches}
    category = dataset_category_row(dataset)
    rows = []
    for idx, row in events.reset_index(drop=True).iterrows():
        match = match_by_event.get(idx, {})
        rows.append({
            "dataset": dataset.name,
            "event_idx": idx,
            "is_true_positive": idx in matched_event_indices,
            "start": float(row.start),
            "end": float(row.end),
            "duration_s": float(row.duration_s),
            "truth_idx": match.get("truth_idx", np.nan),
            "iou": match.get("iou", 0.0),
            "onset_error_s": match.get("onset_error_s", np.nan),
            "duration_error_s": match.get("duration_error_s", np.nan),
            "peak_synchrony": float(row.get("peak_synchrony", np.nan)),
            "participation": float(row.get("participation", np.nan)),
            "llr_aggregate": float(row.get("llr_aggregate", np.nan)),
            "composite_peak": float(row.get("composite_peak", np.nan)),
            "composite_mean": float(row.get("composite_mean", np.nan)),
            "ff_peak": float(row.get("ff_peak", np.nan)),
            **category,
        })
    return pd.DataFrame(rows)


def run_detector_method(dataset: SyntheticDataset, method: str, config):
    if method.startswith("iterative"):
        return compute_iterative_bursts(dataset.spike_times, config=config)
    if method == "traditional":
        return compute_network_bursts(dataset.spike_times, config=config)
    raise ValueError(f"Unknown detector method: {method}")


def run_all_detectors(
    datasets: list[SyntheticDataset],
    detector_configs: dict[str, object],
) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    results = {}
    score_rows = []
    quality_rows = []
    event_quality_tables = []
    for method, config in detector_configs.items():
        results[method] = {}
        for dataset in datasets:
            category = dataset_category_row(dataset)
            try:
                result = run_detector_method(dataset, method, config)
                results[method][dataset.name] = result
                score = score_events_against_truth(result.network_bursts, dataset.burst_intervals, dataset.duration_s)
                quality = summarize_event_quality(result.network_bursts)
                is_real_data = bool(dataset.params.get("is_real_data", False))
                negative_control = (len(dataset.burst_intervals) == 0) and not is_real_data
                score_rows.append({
                    "method": method,
                    "dataset": dataset.name,
                    "type": dataset.culture_type,
                    "units": len(dataset.spike_times),
                    "baseline_rate_median_hz": dataset.params.get("baseline_rate_median_hz", np.nan),
                    "mean_truth_duration_s": np.mean([e - s for s, e in dataset.burst_intervals]) if dataset.burst_intervals else 0.0,
                    **category,
                    **score,
                    "negative_control_overcalled": negative_control and score["detected_network_bursts"] > 0,
                    "burst_truth_missed": (not negative_control) and score["false_negative_events"] > 0,
                    "burstlets": len(result.burstlets),
                    "superbursts": len(result.superbursts),
                    "iterations": result.diagnostics.get("n_iterations", np.nan),
                    "convergence_delta": result.diagnostics.get("convergence_delta", np.nan),
                    "adaptive_bin_ms": result.diagnostics.get("adaptive_bin_ms", np.nan),
                    "composite_threshold": result.diagnostics.get("composite_threshold", np.nan),
                    "participation_threshold": result.plot_data.get("participation_threshold", np.nan),
                    "burst_modulation_index": result.diagnostics.get("burst_modulation_index", np.nan),
                    "burst_activity_detected": result.diagnostics.get("burst_activity_detected", None),
                    "cluster_separation": result.diagnostics.get("cluster_separation", np.nan),
                    "cluster_n_kept": result.diagnostics.get("cluster_n_kept", np.nan),
                })
                quality_rows.append({"method": method, "dataset": dataset.name, **category, **quality})
                event_quality_tables.append(build_event_quality_table(dataset, result).assign(method=method))
            except (IterativeBurstError, BurstDetectorError) as exc:
                score_rows.append({
                    "method": method,
                    "dataset": dataset.name,
                    "type": dataset.culture_type,
                    "units": len(dataset.spike_times),
                    **category,
                    "error": str(exc),
                })
    event_quality = pd.concat(event_quality_tables, ignore_index=True) if event_quality_tables else pd.DataFrame()
    return results, pd.DataFrame(score_rows), pd.DataFrame(quality_rows), event_quality


detector_configs = {
    "iterative": iterative_config,
    "iterative_old": iterative_config_old,
    "traditional": traditional_config,
}
results, detection_summary, quality_summary, event_quality = run_all_detectors(datasets, detector_configs)
detection_summary.sort_values(["expected", "unit_count_tier", "baseline_rate_tier", "morphology", "dataset", "method"])


## Debug Run on Real Datasets

Prints per-iteration convergence, gate decision, and GMM clustering outcome for each real recording.

In [ ]:
real_datasets_list = [d for d in datasets if d.params.get('is_real_data', False)]
for ds in real_datasets_list:
    print(f"\n{'='*65}")
    print(f"Dataset: {ds.name}  ({len(ds.spike_times)} units, {ds.duration_s:.0f} s)")
    _result = compute_iterative_bursts(ds.spike_times, config=iterative_config, debug=True)
    _diag = _result.diagnostics
    print(f"  burst_activity_detected : {_diag['burst_activity_detected']}")
    print(f"  burst_modulation_index  : {_diag['burst_modulation_index']:.3f}  (threshold = {iterative_config.min_burst_modulation})")
    print(f"  cluster_separation      : {_diag.get('cluster_separation')}")
    print(f"  cluster_n_kept          : {_diag.get('cluster_n_kept')}")
    print(f"  network_bursts detected : {len(_result.network_bursts)}")


## Quantification of Detector Quality Metrics

The evaluation compares two methods: **iterative** and **traditional**. It separates three questions:

- **Event count quality:** precision, recall, F1, count error, false-positive event rate, and missed-event rate.
- **Temporal quality:** matched-event IoU, coverage precision/recall/F1, onset/offset/duration error, false-positive time, and missed truth time.
- **Detector signal quality:** event-level LLR/composite/FF where available. Traditional detector rows will have zeros or missing values for iterative-only quality columns.

`detection_summary` is one row per dataset and method. `quality_summary` aggregates detector output columns per dataset and method. `event_quality` keeps one row per predicted network burst with true-positive status and timing errors when matched to ground truth.

In [ ]:

display_cols = [
    "method", "dataset", "benchmark_group", "unit_count_tier", "baseline_rate_tier",
    "burst_duration_tier", "morphology", "recruitment_tier", "precision", "recall", "f1",
    "coverage_f1", "mean_iou", "count_error", "false_positive_rate_per_min", "missed_rate",
    "median_abs_onset_error_s", "median_abs_duration_error_s",
]
detection_summary.sort_values(["expected", "unit_count_tier", "baseline_rate_tier", "morphology", "dataset", "method"])[display_cols]


## Metric Plots

The first figure summarizes detection accuracy and detector quality metrics across all synthetic datasets. The second figure emphasizes evaluation failure modes: count error, temporal coverage, missed truth time, and quality-metric separation between true and false predicted events. Figures are saved into `FIGURE_OUTPUT_DIR`.

In [ ]:

def save_html(fig: go.Figure, filename: str) -> Path:
    path = FIGURE_OUTPUT_DIR / filename
    plotlyjs = True if PLOTLY_OFFLINE else 'cdn'
    fig.write_html(str(path), include_plotlyjs=plotlyjs)
    print(f'Saved: {path}')
    return path

# Alias for callers that use the old name
save_metric_figure = save_html


METHOD_COLORS = {"iterative": "#2563eb", "iterative_old": "#93c5fd", "traditional": "#f97316"}
METHOD_OFFSETS = {"iterative": -0.25, "iterative_old": 0.0, "traditional": 0.25}
CATEGORY_ORDER = [
    "expected", "unit_count_tier", "baseline_rate_tier", "burst_duration_tier", "morphology", "recruitment_tier", "dataset", "method",
]


def ordered_metric_frame(detection_summary: pd.DataFrame, quality_summary: pd.DataFrame) -> pd.DataFrame:
    plot_df = detection_summary.merge(quality_summary, on=["method", "dataset"], how="left", suffixes=("", "_quality"))
    existing = [col for col in CATEGORY_ORDER if col in plot_df.columns]
    return plot_df.sort_values(existing, ascending=True).reset_index(drop=True)


def method_pivot(df: pd.DataFrame, value: str, index: str = "dataset") -> pd.DataFrame:
    return df.pivot_table(index=index, columns="method", values=value, aggfunc="mean")


def category_metric_frame(plot_df: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "precision", "recall", "f1", "coverage_f1", "mean_iou", "count_error",
        "false_positive_rate_per_min", "missed_rate", "false_positive_time_s", "missed_truth_time_s",
    ]
    return (
        plot_df.groupby(["method", "expected", "unit_count_tier", "baseline_rate_tier", "burst_duration_tier", "morphology"], dropna=False)[metrics]
        .mean()
        .reset_index()
        .assign(
            category=lambda df: (
                df["expected"]
                + " | " + df["unit_count_tier"]
                + " | " + df["baseline_rate_tier"] + " rate"
                + " | " + df["burst_duration_tier"]
                + " | " + df["morphology"]
            )
        )
    )



## Real Dataset Comparison: Old Iterative vs New Iterative vs Traditional

In [ ]:
real_mask = detection_summary['type'] == 'real_spikesorted'
real_df = detection_summary[real_mask].copy()

primary_cols = [
    'method', 'dataset',
    'detected_network_bursts',
    'burst_modulation_index',
    'burst_activity_detected',
    'cluster_separation',
    'cluster_n_kept',
]
_primary_cols = [c for c in primary_cols if c in real_df.columns]
print('=== Event Count & Gate/Clustering Diagnostics ===')
display(
    real_df[_primary_cols]
    .sort_values(['dataset', 'method'])
    .reset_index(drop=True)
)

real_quality = quality_summary[quality_summary['dataset'].isin(real_df['dataset'].unique())].copy()
quality_cols = [
    'method', 'dataset',
    'composite_peak_median', 'llr_aggregate_median',
    'participation_median', 'duration_s_median',
]
_quality_cols = [c for c in quality_cols if c in real_quality.columns]
if _quality_cols:
    print('\n=== Event Quality Metrics (medians) ===')
    display(
        real_quality[_quality_cols]
        .sort_values(['dataset', 'method'])
        .reset_index(drop=True)
    )


## Diagnostic Bar Chart: Old Iterative vs New Iterative vs Traditional

In [ ]:
# Real dataset comparison — Plotly grouped bar (3 metrics)
_real = detection_summary[detection_summary['type'] == 'real_spikesorted'].copy()
_datasets_real = sorted(_real['dataset'].unique())
_methods_plot  = [m for m in ['iterative_old', 'iterative', 'traditional']
                  if m in _real['method'].unique()]
_method_colors = {'iterative': '#2563eb', 'iterative_old': '#93c5fd', 'traditional': '#f97316'}

_metrics = [
    ('detected_network_bursts', 'Network Burst Count'),
    ('burst_modulation_index',  f'Burst Modulation Index (gate thr={iterative_config.min_burst_modulation})'),
    ('cluster_separation',      'Cluster Separation'),
]

fig_real = make_subplots(rows=1, cols=3, subplot_titles=[m[1] for m in _metrics])
for _m, (_col_key, _col_title) in enumerate(_metrics, 1):
    for _method in _methods_plot:
        _sub = _real[_real['method'] == _method].set_index('dataset').reindex(_datasets_real)
        _vals = _sub[_col_key].values if _col_key in _sub.columns else [0]*len(_datasets_real)
        fig_real.add_trace(go.Bar(
            name=_method, x=_datasets_real, y=_vals,
            marker_color=_method_colors.get(_method, '#888'),
            hovertemplate='%{x}<br>' + _col_key + '=%{y:.3f}<extra>' + _method + '</extra>',
            legendgroup=_method,
            showlegend=(_m == 1),
        ), row=1, col=_m)

fig_real.update_layout(barmode='group',
                       title='Real Dataset: Old Iterative vs New Iterative vs Traditional',
                       height=420)
fig_real.show()
save_html(fig_real, 'real_comparison_bar.html')


## All-Dataset Overview

Ground-truth bursts are filled green bands. Ground-truth silence periods are filled blue bands. Predicted iterative network bursts are red hatched outlines. Traditional detector predictions are shown separately in the method-comparison metric plots and can be inspected by changing `METHOD_FOR_RASTER` below. The trace below each raster is the selected detector's composite or participation signal, normalized per dataset for visual comparison. The figure is saved into `FIGURE_OUTPUT_DIR`.

In [ ]:



TRUTH_BURST_COLOR = "#16a34a"
TRUTH_SILENCE_COLOR = "#0284c7"
ITERATIVE_COLOR = "#2563eb"
TRADITIONAL_COLOR = "#f97316"
PREDICTED_BURST_COLOR = ITERATIVE_COLOR  # kept for add_reference_lines



def save_figure(fig, filename: str) -> Path:
    path = FIGURE_OUTPUT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path


def add_interval_lane(ax, intervals: list[tuple[float, float]], lane_y: float, color: str, alpha: float = 0.75) -> None:
    if not intervals:
        return
    spans = [(float(s), float(e - s)) for s, e in intervals if e > s]
    if spans:
        ax.broken_barh(spans, (lane_y - 0.32, 0.64), facecolors=color, edgecolors="none", alpha=alpha)


def predicted_intervals(events: pd.DataFrame) -> list[tuple[float, float]]:
    if events.empty:
        return []
    return [(float(row.start), float(row.end)) for _, row in events.reset_index(drop=True).iterrows()]


def plot_truth_prediction_timeline(
    ax,
    dataset: SyntheticDataset,
    iterative_events: pd.DataFrame | None = None,
    traditional_events: pd.DataFrame | None = None,
) -> None:
    # Four lanes: truth bursts (top), truth silences, iterative, traditional (bottom)
    add_interval_lane(ax, dataset.burst_intervals, 3.0, TRUTH_BURST_COLOR)
    add_interval_lane(ax, dataset.silence_intervals, 2.0, TRUTH_SILENCE_COLOR, alpha=0.55)
    if iterative_events is not None:
        add_interval_lane(ax, predicted_intervals(iterative_events), 1.0, ITERATIVE_COLOR, alpha=0.70)
    if traditional_events is not None:
        add_interval_lane(ax, predicted_intervals(traditional_events), 0.0, TRADITIONAL_COLOR, alpha=0.70)
    ax.set_ylim(-0.6, 3.6)
    ax.set_yticks([3.0, 2.0, 1.0, 0.0])
    ax.set_yticklabels(["truth", "silence", "iterative", "traditional"], fontsize=7)
    ax.grid(axis="x", color="#e5e7eb", linewidth=0.6)
    ax.set_axisbelow(True)


def add_reference_lines(ax, dataset: SyntheticDataset, events: pd.DataFrame | None) -> None:
    for s, _ in dataset.burst_intervals:
        ax.axvline(s, color=TRUTH_BURST_COLOR, alpha=0.20, lw=0.8, zorder=0)
    if events is not None and not events.empty:
        for _, row in events.reset_index(drop=True).iterrows():
            ax.axvline(float(row.start), color=PREDICTED_BURST_COLOR, alpha=0.16, lw=0.8, zorder=0)


def plot_overview(datasets: list, results: dict, method: str = METHOD_FOR_RASTER) -> None:
    """Overview: composite/participation signal + event timeline, one dataset per row pair."""
    iter_results = results.get('iterative', {})
    trad_results = results.get('traditional', {})
    n_ds = len(datasets)
    if n_ds == 0:
        return

    fig = make_subplots(
        rows=n_ds * 2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.01,
        row_heights=[0.7, 0.3] * n_ds,
    )

    for idx, dataset in enumerate(datasets):
        _row_sig  = idx * 2 + 1   # signal row
        _row_lane = idx * 2 + 2   # timeline lane row
        iter_result = iter_results.get(dataset.name)
        trad_result = trad_results.get(dataset.name)
        iter_events = iter_result.network_bursts if iter_result is not None else None
        trad_events = trad_result.network_bursts if trad_result is not None else None

        # Signal traces
        if iter_result is not None:
            t   = iter_result.plot_data['t']
            sig_key = 'composite_signal' if 'composite_signal' in iter_result.plot_data else 'participation_signal'
            sig = np.asarray(iter_result.plot_data[sig_key], dtype=float)
            span = np.nanmax(sig) - np.nanmin(sig)
            y = (sig - np.nanmin(sig)) / span if span > 1e-12 else sig * 0.0
            fig.add_trace(go.Scatter(
                x=t, y=y, mode='lines',
                line=dict(color=ITERATIVE_COLOR, width=0.9),
                name='iterative', legendgroup='iterative', showlegend=(idx==0),
                hovertemplate='t=%{x:.2f}s  score=%{y:.3f}<extra>iterative</extra>',
            ), row=_row_sig, col=1)

        if trad_result is not None and 'participation_signal' in trad_result.plot_data:
            t   = trad_result.plot_data['t']
            sig = np.asarray(trad_result.plot_data['participation_signal'], dtype=float)
            span = np.nanmax(sig) - np.nanmin(sig)
            y = (sig - np.nanmin(sig)) / span if span > 1e-12 else sig * 0.0
            fig.add_trace(go.Scatter(
                x=t, y=y, mode='lines',
                line=dict(color=TRADITIONAL_COLOR, width=0.9, dash='dot'),
                name='traditional', legendgroup='traditional', showlegend=(idx==0),
                opacity=0.7,
                hovertemplate='t=%{x:.2f}s  score=%{y:.3f}<extra>traditional</extra>',
            ), row=_row_sig, col=1)

        # Y-axis label = dataset name
        fig.update_yaxes(title_text=dataset.name[:18], row=_row_sig, col=1)

        # Timeline lanes (truth burst / truth silence / predictions)
        _dur = dataset.duration_s
        _lane_data = [
            (dataset.burst_intervals,                      TRUTH_BURST_COLOR,   'truth burst',   3, True),
            (dataset.silence_intervals,                    TRUTH_SILENCE_COLOR, 'truth silence', 2, True),
            (predicted_intervals(iter_events) if iter_events is not None else [], ITERATIVE_COLOR, 'iterative', 1, True),
            (predicted_intervals(trad_events) if trad_events is not None else [], TRADITIONAL_COLOR, 'traditional', 0, True),
        ]
        for _ivs, _col, _lbl, _lane_y, _do in _lane_data:
            for _s, _e in _ivs:
                fig.add_vrect(
                    x0=float(_s), x1=float(_e),
                    row=_row_lane, col=1,
                    fillcolor=_col, opacity=0.65, layer='below', line_width=0,
                )

    fig.update_layout(
        title='Iterative (solid) vs traditional (dashed) — normalised signals + event lanes',
        hovermode='x unified',
        height=max(400, 80 * n_ds),
        legend_tracegroupgap=2,
    )
    fig.update_xaxes(title_text='time (s)', row=n_ds*2, col=1)
    fig.show()
    saved = save_html(fig, 'all-all-both-detectors-overview.html')
    print(f'Saved overview figure: {saved}')

plot_overview(datasets, results, METHOD_FOR_RASTER)


## Detailed Visual Checks

The detailed view combines spike rasters, ground-truth intervals, detector event spans, and the detector's internal signals. Truth uses filled bands; predictions use red hatched outlines. Change `datasets_to_plot` to inspect other cases. Each detailed figure is saved into `FIGURE_OUTPUT_DIR`.


In [ ]:
def units_sorted_by_firing_rate(dataset, descending: bool = True) -> list:
    rates = unit_firing_rates(dataset)
    return sorted(dataset.spike_times.keys(),
                  key=lambda u: rates.get(u, 0.0), reverse=descending)

def unit_firing_rates(dataset) -> dict:
    return {u: len(dataset.spike_times[u]) / dataset.duration_s
            for u in dataset.spike_times}

def predicted_intervals(events) -> list:
    if events is None or (hasattr(events, 'empty') and events.empty):
        return []
    return [(float(row.start), float(row.end)) for _, row in events.reset_index(drop=True).iterrows()]


def _bin_edges_from_centers(t: np.ndarray) -> tuple[np.ndarray, float]:
    """Recover uniform bin edges from bin-center array `t`."""
    t = np.asarray(t, dtype=float)
    if t.size < 2:
        # Degenerate: fabricate a 1 ms bin so downstream math doesn't divide by zero.
        bs = 1e-3
        return np.array([t[0] - bs / 2, t[0] + bs / 2]), bs
    bs = float(t[1] - t[0])
    edges = np.concatenate(([t[0] - bs / 2.0], t + bs / 2.0))
    return edges, bs


def plot_dataset_detail(
    dataset,
    iterative_result,
    traditional_result=None,
    method: str = METHOD_FOR_RASTER,
    *,
    show_truth_prediction_timeline: bool = False,
    show_signals: bool = False,
) -> go.Figure:
    """Spike raster (rich per-spike hover) + predicted-burst ribbon.

    Optional opt-in rows:
        show_truth_prediction_timeline: adds the legacy 4-lane truth/iter/trad timeline.
        show_signals: adds composite/participation/FF/LLR/rate panels (iterative only).
    """
    ordered_units = units_sorted_by_firing_rate(dataset, descending=False)
    rates = unit_firing_rates(dataset)
    iter_events = iterative_result.network_bursts if iterative_result is not None else None
    trad_events = traditional_result.network_bursts if traditional_result is not None else None

    # --- Recover the detector's bin grid for hover annotations -----------------
    pd_i = iterative_result.plot_data if iterative_result is not None else {}
    t_centers = np.asarray(pd_i.get('t', []), dtype=float)
    ff_signal = np.asarray(pd_i.get('ff_signal', []), dtype=float) if 't' in pd_i else np.empty(0)
    llr_signal = np.asarray(pd_i.get('llr_signal', []), dtype=float) if 't' in pd_i else np.empty(0)
    have_bin_info = t_centers.size > 1 and ff_signal.size == t_centers.size and llr_signal.size == t_centers.size

    if have_bin_info:
        bin_edges, bin_size = _bin_edges_from_centers(t_centers)
        n_bins = t_centers.size
    else:
        bin_edges, bin_size, n_bins = None, None, 0

    # --- Plan the row layout ---------------------------------------------------
    # Each entry: (title, height)
    rows: list[tuple[str, float]] = [
        ('Spike raster', 3.0),
        ('Predicted bursts', 0.4),
    ]
    timeline_row = signal_start_row = None
    if show_truth_prediction_timeline:
        timeline_row = len(rows) + 1
        rows.append(('Truth / Prediction timeline', 0.9))
    if show_signals and iterative_result is not None:
        n_trace_rows = 5 if method == 'iterative' else 2
        signal_start_row = len(rows) + 1
        signal_titles = ['Composite', 'Participation', 'FF', 'LLR', 'Rate'][:n_trace_rows]
        for _ttl in signal_titles:
            rows.append((_ttl, 1.0))

    n_rows = len(rows)
    fig = make_subplots(
        rows=n_rows, cols=1,
        shared_xaxes=True,
        row_heights=[h for _, h in rows],
        vertical_spacing=0.02,
        subplot_titles=[t for t, _ in rows],
    )

    # --- Row 1: spike raster with per-spike rich hover -------------------------
    _rate_arr = np.array([rates.get(u, 0.0) for u in ordered_units])
    _rate_span = max(_rate_arr.max() - _rate_arr.min(), 1e-9)
    _rate_norm = (_rate_arr - _rate_arr.min()) / _rate_span

    for _y, _u in enumerate(ordered_units):
        _spks = np.asarray(dataset.spike_times[_u])
        if _spks.size == 0:
            continue
        _col = px.colors.sample_colorscale('Viridis', float(_rate_norm[_y]))[0]
        _unit_rate = float(rates.get(_u, 0.0))

        if have_bin_info:
            _idx = np.clip(
                np.searchsorted(bin_edges, _spks, side='right') - 1,
                0, n_bins - 1,
            )
            _counts = np.bincount(_idx, minlength=n_bins)
            _customdata = np.column_stack([
                np.full(_spks.size, _unit_rate),
                _counts[_idx].astype(float),
                ff_signal[_idx],
                llr_signal[_idx],
            ])
            _hover = (
                'unit %{text}<br>'
                't = %{x:.3f} s<br>'
                'unit rate = %{customdata[0]:.2f} Hz<br>'
                'count in bin = %{customdata[1]:.0f}<br>'
                'FF = %{customdata[2]:.2f}<br>'
                'LLR = %{customdata[3]:.2f}'
                '<extra></extra>'
            )
        else:
            _customdata = np.full((_spks.size, 1), _unit_rate)
            _hover = (
                'unit %{text}<br>'
                't = %{x:.3f} s<br>'
                'unit rate = %{customdata[0]:.2f} Hz'
                '<extra></extra>'
            )

        fig.add_trace(go.Scatter(
            x=_spks, y=np.full_like(_spks, _y, dtype=float),
            mode='markers', marker=dict(color=_col, size=2, opacity=0.8),
            text=[str(_u)] * _spks.size,
            customdata=_customdata,
            hovertemplate=_hover,
            name=str(_u), legendgroup='raster', showlegend=False,
        ), row=1, col=1)
    fig.update_yaxes(title_text='unit (by firing rate)', row=1, col=1,
                     showticklabels=False)

    # --- Row 2: predicted-burst ribbon -----------------------------------------
    fig.update_yaxes(range=[0, 1], showticklabels=False, showgrid=False,
                     zeroline=False, row=2, col=1)
    # Anchor trace so the row renders even without bursts.
    fig.add_trace(go.Scatter(
        x=[0.0, float(dataset.duration_s)],
        y=[0.5, 0.5],
        mode='markers', marker=dict(opacity=0),
        showlegend=False, hoverinfo='skip',
    ), row=2, col=1)
    for _s, _e in predicted_intervals(iter_events):
        fig.add_vrect(
            x0=float(_s), x1=float(_e), row=2, col=1,
            fillcolor=ITERATIVE_COLOR, opacity=0.55, layer='below', line_width=0,
        )

    # --- Optional row: truth / prediction timeline -----------------------------
    if timeline_row is not None:
        for _s, _e in dataset.burst_intervals:
            fig.add_vrect(x0=float(_s), x1=float(_e), row=timeline_row, col=1,
                          fillcolor=TRUTH_BURST_COLOR, opacity=0.7, layer='below', line_width=0)
        for _s, _e in dataset.silence_intervals:
            fig.add_vrect(x0=float(_s), x1=float(_e), row=timeline_row, col=1,
                          fillcolor=TRUTH_SILENCE_COLOR, opacity=0.55, layer='below', line_width=0)
        for _s, _e in predicted_intervals(iter_events):
            fig.add_vrect(x0=float(_s), x1=float(_e), row=timeline_row, col=1,
                          fillcolor=ITERATIVE_COLOR, opacity=0.5, layer='above', line_width=0)
        for _s, _e in predicted_intervals(trad_events):
            fig.add_vrect(x0=float(_s), x1=float(_e), row=timeline_row, col=1,
                          fillcolor=TRADITIONAL_COLOR, opacity=0.45, layer='above', line_width=0)
        fig.update_yaxes(range=[0, 1], showticklabels=False, showgrid=False,
                         zeroline=False, row=timeline_row, col=1)

    # --- Optional rows: signal panels (iterative only) -------------------------
    if signal_start_row is not None:
        t_i = pd_i.get('t')
        if t_i is not None:
            _signal_specs = [
                ('composite_signal',     'composite',     'black'),
                ('participation_signal', 'participation', ITERATIVE_COLOR),
                ('ff_signal',            'FF',            '#7c3aed'),
                ('llr_signal',           'LLR',           '#059669'),
                ('rate_signal',          'rate',          '#dc2626'),
            ][: (5 if method == 'iterative' else 2)]
            for _i, (_key, _lbl, _col) in enumerate(_signal_specs):
                if _key not in pd_i:
                    continue
                _row = signal_start_row + _i
                fig.add_trace(go.Scatter(
                    x=t_i, y=np.asarray(pd_i[_key], dtype=float),
                    mode='lines', line=dict(color=_col, width=0.9),
                    name=_lbl, showlegend=False,
                    hovertemplate='t=%{x:.2f}s  %{y:.3f}<extra>' + _lbl + '</extra>',
                ), row=_row, col=1)

    # --- Layout ----------------------------------------------------------------
    fig.update_layout(
        title=f'{dataset.name} — {dataset.description[:60]}',
        hovermode='closest',  # one spike at a time
        height=max(420, 80 * n_rows + 200),
        showlegend=False,
    )
    # Rangeslider on the last x-axis.
    last_xaxis = 'xaxis' if n_rows == 1 else f'xaxis{n_rows}'
    fig.update_layout(**{last_xaxis: dict(
        title='time (s)',
        rangeslider=dict(visible=True, thickness=0.05),
    )})
    return fig


# Show detail for datasets selected by METHOD_FOR_RASTER
_iter_res = results.get('iterative', {})
_trad_res = results.get('traditional', {})
_show_ds  = [d for d in datasets if d.name in _iter_res or d.name in _trad_res]
for _ds in _show_ds[:6]:  # cap at 6 to avoid very slow execution
    _fig = plot_dataset_detail(
        _ds,
        _iter_res.get(_ds.name),
        _trad_res.get(_ds.name),
    )
    _fig.show()
    save_html(_fig, f'detail_{_ds.name}.html')


## Event Tables

Use these tables to inspect timing and quality columns for any dataset. `network_bursts` is usually the first table to inspect because it is the main per-culture burst call.


In [ ]:
# Fall back to the first available dataset if the named one isn't in results
# (e.g. when USE_SIMULATED_DATA=False and only real data was run).
_available = list(results.get(METHOD_FOR_TABLE, {}).keys())
if DATASET_FOR_TABLE not in results.get(METHOD_FOR_TABLE, {}):
    if not _available:
        raise KeyError(f"No results found for method '{METHOD_FOR_TABLE}'")
    DATASET_FOR_TABLE = _available[0]
    print(f"Note: falling back to first available dataset: {DATASET_FOR_TABLE}")

selected = results[METHOD_FOR_TABLE][DATASET_FOR_TABLE]

print(f"Diagnostics for {METHOD_FOR_TABLE} / {DATASET_FOR_TABLE}")
for key in [
    "n_iterations", "convergence_delta", "adaptive_bin_ms", "biological_isi_s",
    "composite_baseline", "composite_threshold", "burstlet_merge_gap_s", "network_merge_gap_s",
]:
    if key in selected.diagnostics:
        print(f"{key}: {selected.diagnostics[key]}")

if "feature_weights_final" in selected.diagnostics:
    print("\nFinal feature weights [PFR, participation, FF..., LLR, burstiness]:")
    print(np.round(selected.diagnostics["feature_weights_final"], 3))

selected.network_bursts

## Interpretation Notes

- The notebook now compares `iterative` and `traditional` detectors on the same synthetic spike trains.
- Independent Poisson datasets are negative controls. If `negative_control_overcalled` is `True`, that method is producing false positives on unconnected spike trains.
- `detection_summary` reports event-count quality, temporal quality, and failure-mode metrics per method and dataset. Use event F1 for count matching and coverage F1 / IoU for temporal overlap.
- `count_error`, `fragmentation_index`, and `merge_flag` help distinguish over-splitting from over-merging.
- `quality_summary` quantifies detector-specific event metrics per method. Iterative-only columns such as composite and LLR are absent or zero for the traditional detector.
- Set `METHOD_FOR_RASTER` or `METHOD_FOR_TABLE` to `"traditional"` to inspect traditional detector rasters or event tables.

## Before / After Summary

In [ ]:
_real = detection_summary[detection_summary['type'] == 'real_spikesorted'].copy()
_pivot = _real.pivot_table(
    index='dataset', columns='method',
    values='detected_network_bursts', aggfunc='first',
)

print('\n' + '='*70)
print('BEFORE/AFTER: Network Burst Event Counts on Real (Unlabelled) Datasets')
print('='*70)
_col_order = [c for c in ['iterative_old', 'iterative', 'traditional'] if c in _pivot.columns]
display(
    _pivot[_col_order].rename(
        columns={'iterative': 'iterative (new)', 'iterative_old': 'iterative (old)'}
    )
)

print('\nBurst Modulation Index (new iterative; gate threshold =', iterative_config.min_burst_modulation, '):')
_bmi_rows = _real[_real['method'] == 'iterative'][['dataset', 'burst_modulation_index', 'burst_activity_detected']]
for _, row in _bmi_rows.iterrows():
    gate = 'PASS' if row['burst_activity_detected'] else 'GATED'
    print(f"  {row['dataset']}: BMI={row['burst_modulation_index']:.2f}  [{gate}]")


## Summary — Updated Iterative Detector vs Traditional

The updated iterative detector adds two post-convergence quality filters:

1. **Burstlet-level LLR gate** — rejects bridge-like burstlets after materialisation instead of deleting an entire heterogeneous recording. It uses `llr_aggregate` on each burstlet; `min_burst_modulation=0.0` disables the gate.

2. **Multi-component GMM clustering** — start with several components, then merge similar components in feature space before discarding the noise-like cluster. This handles over-segmentation better than a fixed 2-component split.

**New `diagnostics` keys returned by `compute_iterative_bursts`:**
- `burst_modulation_index` — burstlet LLR score summary
- `burst_activity_detected` — whether burstlets survived the final filters
- `cluster_separation` — merged-component separation (None if skipped)
- `cluster_n_kept` — number of burstlets retained after clustering
